# OSTutor - QLoRA fine-tuning on a free GPU
Runtime > Change runtime type > **T4 GPU** (or Kaggle P100). Takes roughly 20-40 minutes for the 8B model on ~650 examples.

In [ ]:
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change the runtime type')

## 1. Get the project
Upload `OSTutorLLM.zip` (left sidebar > Files) or clone your repository.

In [ ]:
!unzip -q -o OSTutorLLM.zip -d /content 2>/dev/null || git clone https://github.com/i-Anurag1/OSTutorLLM /content/OSTutorLLM
%cd /content/OSTutorLLM

In [ ]:
!pip install -q -r training/requirements-train.txt

## 2. Rebuild and inspect the original dataset (optional - the processed files already ship in the zip)

In [ ]:
!python training/build_dataset.py
!python training/test_training.py

In [ ]:
import json; print(json.dumps(json.load(open('training/data/processed/preprocessing_report.json'))['stages'], indent=1)[:1500])

## 3. Train (assistant-only loss, 4-bit base, LoRA r=16)

In [ ]:
!python training/train_qlora.py --base Qwen/Qwen3-8B --epochs 3 --out training/outputs/ostutor-lora

## 4. Merge the adapter and convert to GGUF for Ollama

In [ ]:
!python training/merge_and_export.py --base Qwen/Qwen3-8B --adapter training/outputs/ostutor-lora --out training/outputs/ostutor-merged

In [ ]:
!git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt
!python /content/llama.cpp/convert_hf_to_gguf.py training/outputs/ostutor-merged --outfile training/outputs/ostutor-merged/ostutor-qwen3-f16.gguf --outtype f16

In [ ]:
# build llama-quantize (CPU build is enough)
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF && cmake --build build --config Release -j --target llama-quantize
!/content/llama.cpp/build/bin/llama-quantize training/outputs/ostutor-merged/ostutor-qwen3-f16.gguf training/outputs/ostutor-merged/ostutor-qwen3-q4_k_m.gguf Q4_K_M

## 5. Download and serve
Download `ostutor-qwen3-q4_k_m.gguf` and `Modelfile`, then on your PC:
```
ollama create ostutor-qwen3 -f Modelfile
```
Set `LLM_MODEL=ostutor-qwen3` in `.env`, restart the API, and compare with `python training/evaluate_models.py --models qwen3:8b ostutor-qwen3`.

In [ ]:
from google.colab import files
files.download('training/outputs/ostutor-merged/ostutor-qwen3-q4_k_m.gguf')
files.download('training/outputs/ostutor-merged/Modelfile')
files.download('training/outputs/ostutor-lora/training_summary.json')